<a href="https://www.kaggle.com/code/aklnmelike/air-quality-predict-ensemble-methods?scriptVersionId=333575519" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/air-quality-and-health-impact-dataset/air_quality_health_impact_data.csv


In [2]:
df = pd.read_csv('/kaggle/input/air-quality-and-health-impact-dataset/air_quality_health_impact_data.csv')
pd.set_option('max_colwidth', 100)
df.head(5)

,RecordID,AQI,PM10,PM2_5,NO2,SO2,O3,Temperature,Humidity,WindSpeed,RespiratoryCases,CardiovascularCases,HospitalAdmissions,HealthImpactScore,HealthImpactClass
0,1,187.270059,295.853039,13.038560,6.639263,66.161150,54.624280,5.150335,84.424344,6.137755,7,5,1,97.244041,0.0
1,2,475.357153,246.254703,9.984497,16.318326,90.499523,169.621728,1.543378,46.851415,4.521422,10,2,0,100.000000,0.0
2,3,365.996971,84.443191,23.111340,96.317811,17.875850,9.006794,1.169483,17.806977,11.157384,13,3,0,100.000000,0.0
3,4,299.329242,21.020609,14.273403,81.234403,48.323616,93.161033,21.925276,99.473373,15.302500,8,8,1,100.000000,0.0
4,5,78.009320,16.987667,152.111623,121.235461,90.866167,241.795138,9.217517,24.906837,14.534733,9,0,1,95.182643,0.0


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [4]:
# Özellikler ve hedef değişkeni belirleme
X = df.drop(columns=['HealthImpactClass'])
y = df['HealthImpactClass']

# Veri setini eğitim ve test olarak ayırma
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=58)

## 1) Boosting

Boosting, zayıf öğrenicileri (weak learners) güçlü bir tahminciye dönüştürmeyi amaçlayan bir makine öğrenimi topluluk (ensemble) tekniğidir. Boosting, her bir zayıf öğrenicinin hatalarını düzelterek modelin doğruluğunu artırır.

Boosting'in prensipleri şunlardır:

- Adım Adım Öğrenme: Boosting, zayıf öğrenicileri ardışık olarak eğitir. Her yeni model, önceki modellerin hatalarını düzelterek daha iyi tahmin yapmaya çalışır.
- Ağırlıklandırma: Her bir örneğin hata oranına göre ağırlıklandırılır. Hatalı sınıflandırılmış örnekler daha yüksek ağırlık alır, böylece sonraki model bu örnekleri daha iyi öğrenir.
- Topluluk Kararı: Nihai tahmin, tüm zayıf öğrenicilerin tahminlerinin ağırlıklı toplamı veya çoğunluk kararı ile belirlenir.

### 1.1 Gradient Boosting

Gradient Boosting, zayıf modellerin ardışık olarak eğitilmesiyle güçlü bir model oluşturur. Her bir model, önceki modellerin hatalarını düzelterek öğrenir.

In [5]:
from sklearn.ensemble import GradientBoostingClassifier

# Gradient Boosting modelini oluşturma
gb_model = GradientBoostingClassifier(random_state=58)
gb_model.fit(X_train, y_train)

# Tahmin yapma
y_pred = gb_model.predict(X_test)

# Sonuçları değerlendirme
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[960   1   2   2   1]
 [  3 104   1   1   2]
 [  5   1  45   1   4]
 [  2   3   1   7   1]
 [  3   6   1   3   3]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99       966
         1.0       0.90      0.94      0.92       111
         2.0       0.90      0.80      0.85        56
         3.0       0.50      0.50      0.50        14
         4.0       0.27      0.19      0.22        16

    accuracy                           0.96      1163
   macro avg       0.71      0.68      0.70      1163
weighted avg       0.96      0.96      0.96      1163



### 1.2 AdaBoost

AdaBoost, zayıf öğreniciler (genellikle decision tree'ler) kullanarak ardışık olarak modeller oluşturur ve her adımda hataları düzelterek güçlü bir model oluşturur.

In [6]:
from sklearn.ensemble import AdaBoostClassifier

# AdaBoost modelini oluşturma
ab_model = AdaBoostClassifier(random_state=58)
ab_model.fit(X_train, y_train)

# Tahmin yapma
y_pred = ab_model.predict(X_test)

# Sonuçları değerlendirme
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[694 153 101  18   0]
 [ 62  37  10   2   0]
 [ 36   5   8   4   3]
 [  8   1   3   1   1]
 [  5   4   5   1   1]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.86      0.72      0.78       966
         1.0       0.18      0.33      0.24       111
         2.0       0.06      0.14      0.09        56
         3.0       0.04      0.07      0.05        14
         4.0       0.20      0.06      0.10        16

    accuracy                           0.64      1163
   macro avg       0.27      0.27      0.25      1163
weighted avg       0.74      0.64      0.68      1163



### 1.3 XGBoost

XGBoost, performans ve hız açısından oldukça güçlü bir boosting algoritmasıdır. Aynı zamanda eksik verilerle başa çıkma yeteneği de vardır.

In [7]:
import xgboost as xgb

# XGBoost modelini oluşturma
xgb_model = xgb.XGBClassifier(random_state=58)
xgb_model.fit(X_train, y_train)

# Tahmin yapma
y_pred = xgb_model.predict(X_test)

# Sonuçları değerlendirme
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[960   0   3   2   1]
 [  2 104   2   2   1]
 [  2   1  44   6   3]
 [  1   0   2  10   1]
 [  5   2   2   3   4]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99       966
         1.0       0.97      0.94      0.95       111
         2.0       0.83      0.79      0.81        56
         3.0       0.43      0.71      0.54        14
         4.0       0.40      0.25      0.31        16

    accuracy                           0.96      1163
   macro avg       0.73      0.74      0.72      1163
weighted avg       0.97      0.96      0.96      1163



## 2) Majority Voting

Majority voting ensemble, birden fazla makine öğrenimi modelinin tahminlerini birleştirerek nihai sınıflandırma kararını veren bir ensemble yöntemidir. Bu yöntemde, her bir modelin verdiği sınıf tahminleri alınır ve en çok oy alan sınıf, nihai tahmin olarak seçilir. Bu yaklaşım, farklı modellerin güçlü yönlerini birleştirerek daha doğru ve sağlam bir tahmin yapmayı amaçlar.

Çalışma Prensibi
Farklı Modellerin Eğitimi: Farklı türde makine öğrenimi modelleri (örneğin, decision tree, k-nearest neighbors, logistic regression) aynı veri seti üzerinde eğitilir.
Tahminlerin Alınması: Test verileri üzerinde her bir modelin tahminleri yapılır.
Oyların Birleştirilmesi: Her bir veri noktası için modellerin yaptığı tahminler alınır ve en çok oyu alan sınıf, nihai tahmin olarak belirlenir.

In [8]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Veri setini yükleyin ve bağımlı ve bağımsız değişkenleri ayır
X = df.drop(columns=['HealthImpactClass'])
y = df['HealthImpactClass']

# Veri setini eğitim ve test olarak ayır
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=58)

# Modelleri tanımla
log_clf = LogisticRegression(random_state=58, max_iter=10000)
knn_clf = KNeighborsClassifier(n_neighbors=5)
dt_clf = DecisionTreeClassifier(random_state=58)

# Majority Voting Ensemble oluştur
voting_clf = VotingClassifier(
    estimators=[('lr', log_clf), ('knn', knn_clf), ('dt', dt_clf)],
    voting='hard'  # 'hard' majority voting, 'soft' probability averaging
)

# Ansambl modelini eğit
voting_clf.fit(X_train, y_train)

# Tahmin yapın
y_pred = voting_clf.predict(X_test)

# Sonuçları değerlendir
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[963   1   0   0   2]
 [ 39  70   0   2   0]
 [ 18  16  21   1   0]
 [  7   3   4   0   0]
 [ 11   3   1   0   1]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.93      1.00      0.96       966
         1.0       0.75      0.63      0.69       111
         2.0       0.81      0.38      0.51        56
         3.0       0.00      0.00      0.00        14
         4.0       0.33      0.06      0.11        16

    accuracy                           0.91      1163
   macro avg       0.56      0.41      0.45      1163
weighted avg       0.89      0.91      0.89      1163



/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 3) Bagging (Bootstrap Aggregating)

Bagging yöntemi, modelin genel performansını artırmak amacıyla birden fazla modelin aynı veri setinden rastgele seçilmiş alt örnekler üzerinde eğitildiği bir ensemble yöntemidir. Bagging, overfitting riskini azaltarak modelin genelleme yeteneğini artırır. Bu yöntem, özellikle yüksek varyansa sahip modellerde etkili sonuçlar verir.
Bu yönteme örnek olarak Random Forest’ı ayrı bir .ipynb dosyasında kullandım. Random Forest da bir bagging yöntemidir.

## 4) Stacking

Stacking, birden fazla modelin tahminlerini bir araya getirerek, nihai tahmini yapmak için bu tahminleri ikinci bir modelle (meta-learner) öğrenen bir ensemble yöntemidir. Stacking, temel öğrenicilerin (base learners) zayıf yanlarını telafi ederek genel model performansını artırmayı amaçlar.

Stacking'in Adımları
Temel Öğrenicilerin Eğitimi (Base Learners):

Farklı algoritmalardan bir dizi temel model (örneğin, lojistik regresyon, SVM, karar ağaçları) eğitilir.
Bu modeller veri setinin farklı özelliklerini yakalamaya çalışır.
Meta Öğrenicinin Eğitimi (Meta-Learner):

Temel öğrenicilerin tahminleri bir araya getirilir ve bu tahminler, yeni bir model (meta-learner) ile birleştirilir.
Meta-learner, temel öğrenicilerin tahminlerine dayanarak nihai tahmini yapar.

Stacking, çok sınıflı veri setlerinde farklı model türlerinin güçlü yönlerini bir araya getirerek daha iyi bir genel performans sağlayabilir. Özellikle bu veri setinde, çeşitli hava kalitesi ve sağlık etkisi ölçümlerine dayanan çok sayıda sınıf bulunduğundan, stacking gibi bir ensemble yöntemi, sınıflandırma performansını artırmada etkili olabilir.

In [9]:
import numpy as np
import pandas as pd
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Veri setini yükleme
# df = pd.read_csv('path_to_dataset.csv')

# Özellikler ve hedef değişkeni belirleme
X = df.drop(columns=['HealthImpactClass'])
y = df['HealthImpactClass']

# Veri setini eğitim ve test olarak ayırma
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=58)

# Temel öğrenicileri tanımlama
estimators = [
    ('lr', LogisticRegression(max_iter=1000, random_state=58)),
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('dt', DecisionTreeClassifier(random_state=58))
]

# StackingClassifier'ı tanımlama
stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000, random_state=58),
    cv=5
)

# Modeli eğitme
stacking_clf.fit(X_train, y_train)

# Tahmin yapma
y_pred = stacking_clf.predict(X_test)

# Sonuçları değerlendirme
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Confusion Matrix:
[[961   2   1   1   1]
 [  4 103   1   2   1]
 [  4   1  47   4   0]
 [  3   2   1   8   0]
 [  4   2   5   3   2]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.98      0.99      0.99       966
         1.0       0.94      0.93      0.93       111
         2.0       0.85      0.84      0.85        56
         3.0       0.44      0.57      0.50        14
         4.0       0.50      0.12      0.20        16

    accuracy                           0.96      1163
   macro avg       0.74      0.69      0.69      1163
weighted avg       0.96      0.96      0.96      1163

